## **Tahap 1: Inisialisasi Library dan Konfigurasi Sistem**

Pada tahap awal ini, kita memuat seluruh *library* Python yang diperlukan untuk membangun sistem *Information Retrieval* berbasis **TF-IDF** dan **Cosine Similarity**.

### **Library yang Digunakan:**
* **Pandas (`pd`)**: Digunakan untuk manajemen dataset dalam bentuk tabel (DataFrame).
* **Numpy (`np`)**: Digunakan untuk operasi vektor matematika dan perhitungan *Cosine Similarity*.
* **Math**: Digunakan untuk menghitung fungsi logaritma pada rumus IDF.
* **Re**: Digunakan untuk *Regular Expression* dalam tahap pembersihan teks (preprocessing).
* **IPython Display**: Digunakan agar *output* tabel di Google Colab terlihat interaktif, rapi, dan profesional.

---

In [ ]:
import pandas as pd
import numpy as np
import math
import re
from IPython.display import display

# Mengatur tampilan plot
%matplotlib inline
sns.set(style="whitegrid")
print("✅ Library berhasil diimpor.")

✅ Library berhasil diimpor.


In [ ]:
# Memuat dataset berantai dari CSV
file_csv = 'dataset_mbg.csv'
data_frame = pd.read_csv(file_csv)

# Menampilkan 5 data pertama untuk memastikan data terbaca
print(f"Total Dokumen: {len(df)}")
df.head()

Total Dokumen: 10


,ID,Isi Dokumen,konten_bersih
0,1,Pemerintah meluncurkan program makan bergizi g...,pemerintah meluncurkan program makan bergizi g...
1,2,Program makan bergizi gratis meningkatkan kual...,program makan bergizi gratis meningkatkan kual...
2,3,Kualitas nutrisi sangat penting untuk pertumbu...,kualitas nutrisi sangat penting untuk pertumbu...
3,4,Seluruh anak sekolah harus mendapatkan asupan ...,seluruh anak sekolah harus mendapatkan asupan ...
4,5,Asupan protein tinggi diperoleh dari konsumsi ...,asupan protein tinggi diperoleh dari konsumsi ...


## **Tahap 2: Pemuatan Dataset dan Pra-pemrosesan Teks (Preprocessing)**

Pada tahap ini, kita memuat dataset utama dari file CSV dan melakukan pembersihan teks. Langkah ini penting untuk memastikan bahwa variasi penulisan (seperti huruf kapital atau simbol) tidak memengaruhi akurasi perhitungan bobot kata.

### **Proses yang dilakukan:**
1. **Pemuatan Data**: Membaca file `dataset_mbg.csv` ke dalam struktur DataFrame.
2. **Lowercasing**: Mengubah semua teks menjadi huruf kecil agar kata "Makan" dan "makan" dianggap sama.
3. **Filtering**: Menghapus angka, tanda baca, dan karakter non-alfabet menggunakan *Regular Expression*.
4. **Normalisasi Spasi**: Menghapus spasi berlebih (trimming) agar pemotongan kata (*tokenizing*) menjadi lebih rapi.



---

In [ ]:
# Fungsi pembersihan teks
def bersihkan_teks(kalimat):
    kalimat = str(kalimat).lower()
    kalimat = re.sub(r'[^a-z0-9\s]', '', kalimat)
    return kalimat

print("✅ Data MBG berhasil dimuat.")
display(data_frame.head(10))

✅ Data MBG berhasil dimuat.


,ID,Isi Dokumen
0,1,Pemerintah meluncurkan program makan bergizi g...
1,2,Program makan bergizi gratis meningkatkan kual...
2,3,Kualitas nutrisi sangat penting untuk pertumbu...
3,4,Seluruh anak sekolah harus mendapatkan asupan ...
4,5,Asupan protein tinggi diperoleh dari konsumsi ...
5,6,Nelayan menyediakan ikan segar untuk satuan pe...
6,7,Satuan pelayanan gizi mengelola distribusi mak...
7,8,Kebersihan dapur menjamin keamanan pangan bagi...
8,9,Seluruh penerima manfaat program diharapkan be...
9,10,Penurunan angka stunting adalah target utama p...


## **Tahap 3: Perhitungan Term Frequency (TF)**

`Term Frequency` (TF) adalah langkah untuk mengukur bobot lokal suatu kata dalam sebuah dokumen. Prinsip dasarnya adalah: semakin sering suatu kata muncul dalam sebuah dokumen, semakin tinggi bobot kata tersebut untuk dokumen itu.

### **Rumus yang Digunakan:**
$$TF(t, d) = \frac{\text{Jumlah kemunculan kata } t \text{ dalam dokumen } d}{\text{Total kata dalam dokumen } d}$$

Langkah-langkah pada tahap ini:
1. **Tokenisasi**: Memecah kalimat menjadi daftar kata tunggal.
2. **Kalkulasi Frekuensi**: Menghitung jumlah kemunculan setiap kata.
3. **Normalisasi**: Membagi frekuensi dengan panjang dokumen agar nilai TF berada dalam rentang yang adil (independen terhadap panjang dokumen).

In [ ]:
# Fungsi hitung TF manual
def kalkulasi_tf(dokumen):
    term = bersihkan_teks(dokumen).split()
    tf_val = {}
    for t in term:
        tf_val[t] = tf_val.get(t, 0) + 1
    return {t: val / len(term) for t, val in tf_val.items()}

# Proses TF untuk semua dokumen
isi_dokumen = data_frame['Isi Dokumen'].tolist()
list_tf = [kalkulasi_tf(d) for d in isi_dokumen]

# Mendapatkan kosakata unik untuk kolom tabel
kosakata = sorted(list(set([kata for doc in isi_dokumen for kata in bersihkan_teks(doc).split()])))

# Membuat Tabel TF
tabel_tf = pd.DataFrame(list_tf, columns=kosakata).fillna(0)
print("=== ANALISIS TERM FREQUENCY (TF) ===")
display(tabel_tf.head(10).style.background_gradient(cmap='Purples').format(precision=3))

=== ANALISIS TERM FREQUENCY (TF) ===


,adalah,anak,angka,asupan,bagi,bebas,bergizi,dapur,dari,diharapkan,diperoleh,distribusi,gizi,gratis,harian,harus,ikan,indonesia,ke,keamanan,kebersihan,konsumsi,kualitas,lokal,makan,makanan,manfaat,meluncurkan,mendapatkan,mengelola,meningkatkan,menjamin,menyediakan,nasional,nelayan,nutrisi,pangan,pelayanan,pemerintah,penerima,penting,penurunan,pertumbuhan,program,protein,sangat,satuan,segar,sekolah,seluruh,siswa,stunting,target,tinggi,untuk,utama
0,0.000,0.000,0.000,0.000,0.000,0.000,0.143,0.000,0.000,0.000,0.000,0.000,0.000,0.143,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.143,0.000,0.000,0.143,0.000,0.000,0.000,0.000,0.000,0.143,0.000,0.000,0.000,0.000,0.143,0.000,0.000,0.000,0.000,0.143,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
1,0.000,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.125,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.000
2,0.000,0.125,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.125,0.000,0.125,0.000,0.000,0.125,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.000,0.125,0.000
3,0.000,0.125,0.000,0.125,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.125,0.125,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.125,0.125,0.000,0.000,0.000,0.000,0.000,0.000
4,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.125,0.000,0.125,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.125,0.000,0.125,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.000
5,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.125,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.125,0.125,0.000,0.000,0.000,0.000,0.000,0.000,0.125,0.000
6,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.125,0.125,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
7,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.125,0.125,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
8,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.125,0.125,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.125,0.000,0.000,0.000,0.000
9,0.125,0.000,0.125,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.125,0.000,0.000,0.125,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.125,0.125,0.000,0.000,0.125


## **Tahap 4: Perhitungan Inverse Document Frequency (IDF)**

`Inverse Document Frequency` (IDF) digunakan untuk menentukan bobot kepentingan global suatu kata di seluruh koleksi dokumen. Semakin langka suatu kata muncul di banyak dokumen, semakin tinggi nilai IDF-nya (dianggap sebagai kata kunci unik).

### **Rumus yang Digunakan:**
$$IDF(t) = \ln\left(\frac{N}{df(t)}\right) + 1$$

**Keterangan:**
* $N$: Total jumlah dokumen dalam dataset (10 dokumen).
* $df(t)$: Jumlah dokumen yang mengandung kata $t$.
* $+ 1$: Berfungsi sebagai *smoothing* agar nilai bobot tetap stabil.

Pada tahap ini, tabel ditampilkan secara **vertikal** agar mempermudah analisis bobot setiap kosakata secara mendalam.

In [ ]:
# Fungsi hitung IDF manual
def kalkulasi_idf(korpus):
    N = len(korpus)
    idf_val = {}
    for k in kosakata:
        # Menghitung Document Frequency (df)
        df_count = sum(1 for d in korpus if k in bersihkan_teks(d).split())
        # Rumus IDF dengan ln + smoothing
        idf_val[k] = math.log(N / df_count) + 1
    return idf_val

# Eksekusi fungsi
bobot_idf = kalkulasi_idf(isi_dokumen)

# Membuat Tabel IDF dan di-Transpose agar vertikal (.T)
tabel_idf_vertikal = pd.DataFrame.from_dict(bobot_idf, orient='index', columns=['Nilai IDF'])

print("=== ANALISIS INVERSE DOCUMENT FREQUENCY (IDF) VERTICAL ===")
# Styling tabel agar terlihat lebih bersih dan profesional
display(tabel_idf_vertikal.style.set_properties(**{
    'background-color': '#f8f9fa',
    'color': '#333333',
    'border-color': 'white'
}).format(precision=3))

=== ANALISIS INVERSE DOCUMENT FREQUENCY (IDF) VERTICAL ===


,Nilai IDF
adalah,3.303
anak,2.609
angka,3.303
asupan,2.609
bagi,3.303
bebas,3.303
bergizi,2.609
dapur,2.609
dari,2.609
diharapkan,3.303


## **Tahap 5: Matriks Pembobotan TF-IDF (Weighting)**

Pada tahap ini, kita melakukan penggabungan antara nilai **Term Frequency (TF)** dan **Inverse Document Frequency (IDF)**. Hasil perkalian ini memberikan bobot akhir yang menunjukkan seberapa relevan sebuah kata terhadap dokumen dalam seluruh koleksi (korpus).

### **Rumus yang Digunakan:**
$$W_{t,d} = TF(t, d) \times IDF(t)$$

**Logika Perhitungan:**
* Jika sebuah kata sering muncul di satu dokumen tapi jarang di dokumen lain (TF tinggi, IDF tinggi), maka nilai TF-IDF akan **sangat besar** (kata kunci kuat).
* Jika sebuah kata muncul di semua dokumen (TF ada, IDF rendah), maka nilai TF-IDF akan **kecil** (kata umum/stopword).

Matriks ini akan digunakan sebagai koordinat atau "vektor" dokumen dalam ruang multidimensi untuk proses pencarian nantinya.

In [ ]:
# Menghitung Vektor TF-IDF (TF * IDF)
matriks_akhir = []
for tf in list_tf:
    baris_bobot = [tf.get(k, 0) * bobot_idf[k] for k in kosakata]
    matriks_akhir.append(baris_bobot)

# Membuat Tabel TF-IDF
tabel_tfidf = pd.DataFrame(matriks_akhir, columns=kosakata)
print("=== MATRIKS PEMBOBOTAN TF-IDF (WEIGHTING) ===")
# Menggunakan warna hijau-kuning agar terlihat kontras
display(tabel_tfidf.head(10).style.background_gradient(cmap='YlGn').format(precision=3))

=== MATRIKS PEMBOBOTAN TF-IDF (WEIGHTING) ===


,adalah,anak,angka,asupan,bagi,bebas,bergizi,dapur,dari,diharapkan,diperoleh,distribusi,gizi,gratis,harian,harus,ikan,indonesia,ke,keamanan,kebersihan,konsumsi,kualitas,lokal,makan,makanan,manfaat,meluncurkan,mendapatkan,mengelola,meningkatkan,menjamin,menyediakan,nasional,nelayan,nutrisi,pangan,pelayanan,pemerintah,penerima,penting,penurunan,pertumbuhan,program,protein,sangat,satuan,segar,sekolah,seluruh,siswa,stunting,target,tinggi,untuk,utama
0,0.000,0.000,0.000,0.000,0.000,0.000,0.373,0.000,0.000,0.000,0.000,0.000,0.000,0.373,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.373,0.000,0.000,0.472,0.000,0.000,0.000,0.000,0.000,0.472,0.000,0.000,0.000,0.000,0.373,0.000,0.000,0.000,0.000,0.315,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
1,0.000,0.000,0.000,0.000,0.000,0.000,0.326,0.000,0.000,0.000,0.000,0.000,0.000,0.326,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.326,0.000,0.326,0.000,0.000,0.000,0.000,0.000,0.413,0.000,0.000,0.000,0.000,0.326,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.275,0.000,0.000,0.000,0.000,0.000,0.000,0.413,0.000,0.000,0.000,0.000,0.000
2,0.000,0.326,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.326,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.326,0.000,0.000,0.000,0.000,0.413,0.000,0.413,0.000,0.000,0.413,0.000,0.000,0.326,0.000,0.000,0.000,0.000,0.000,0.326,0.000
3,0.000,0.326,0.000,0.326,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.413,0.413,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.413,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.326,0.000,0.000,0.000,0.326,0.326,0.000,0.000,0.000,0.000,0.000,0.000
4,0.000,0.000,0.000,0.326,0.000,0.000,0.000,0.000,0.326,0.000,0.413,0.000,0.000,0.000,0.000,0.000,0.326,0.000,0.000,0.000,0.000,0.413,0.000,0.413,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.326,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.413,0.000,0.000
5,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.326,0.000,0.000,0.000,0.326,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.413,0.000,0.413,0.000,0.000,0.326,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.326,0.413,0.000,0.000,0.000,0.000,0.000,0.000,0.326,0.000
6,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.326,0.000,0.000,0.000,0.413,0.326,0.000,0.000,0.000,0.000,0.000,0.413,0.000,0.000,0.000,0.000,0.000,0.000,0.413,0.000,0.000,0.000,0.413,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.326,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.326,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
7,0.000,0.000,0.000,0.000,0.413,0.000,0.000,0.326,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.413,0.413,0.000,0.000,0.000,0.000,0.000,0.326,0.000,0.000,0.000,0.000,0.413,0.000,0.000,0.000,0.000,0.413,0.000,0.000,0.326,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
8,0.000,0.000,0.000,0.000,0.000,0.413,0.000,0.000,0.326,0.413,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.326,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.326,0.000,0.000,0.000,0.275,0.000,0.000,0.000,0.000,0.000,0.326,0.000,0.326,0.000,0.000,0.000,0.000
9,0.413,0.000,0.413,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.413,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.326,0.000,0.000,0.413,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.326,0.413,0.000,0.000,0.413


## **Tahap 6: Sistem Pencarian dan Cosine Similarity**

Pada tahap akhir ini, kita membangun mesin pencari (*search engine*) sederhana. Ketika pengguna memasukkan kata kunci (kueri), sistem akan melakukan langkah-langkah berikut:

1. **Vektorisasi Kueri**: Mengubah kueri pengguna menjadi vektor TF-IDF menggunakan kosakata dan bobot IDF yang telah dihitung sebelumnya.
2. **Perhitungan Cosine Similarity**: Menghitung sudut kemiripan antara vektor kueri dengan setiap vektor dokumen dalam matriks TF-IDF.
3. **Ranking**: Mengurutkan dokumen dari skor tertinggi (paling mirip) ke terendah.

### **Rumus Cosine Similarity:**
$$\text{similarity}(Q, D) = \frac{Q \cdot D}{\|Q\| \|D\|}$$

**Keterangan:**
* $Q \cdot D$: *Dot product* antara vektor kueri dan vektor dokumen.
* $\|Q\| \|D\|$: Perkalian panjang (norma) dari kedua vektor.
* **Skor 1.0**: Dokumen identik dengan kueri.
* **Skor 0.0**: Dokumen tidak memiliki keterkaitan sama sekali.

In [ ]:
def cari_relevansi(kueri, top_n=3):
    # Vektorisasi Kueri
    q_tf = kalkulasi_tf(kueri)
    q_vec = np.array([q_tf.get(k, 0) * bobot_idf.get(k, 0) for k in kosakata])

    sim_scores = []
    for d_vec in matriks_akhir:
        d_vec = np.array(d_vec)
        # Rumus Cosine Similarity
        dot = np.dot(q_vec, d_vec)
        norm_q = np.linalg.norm(q_vec)
        norm_d = np.linalg.norm(d_vec)

        skor = dot / (norm_q * norm_d) if (norm_q > 0 and norm_d > 0) else 0
        sim_scores.append(round(skor, 4))

    # Membuat Laporan Hasil
    hasil_final = data_frame.copy()
    hasil_final['Skor_Kemiripan'] = sim_scores
    hasil_final = hasil_final.sort_values(by='Skor_Kemiripan', ascending=False).head(top_n)

    # Tambahkan kolom Peringkat
    hasil_final.insert(0, 'Peringkat', range(1, len(hasil_final) + 1))
    return hasil_final

# --- AREA INPUT USER ---
print("\n" + "—"*45)
kata_kunci = input("Masukkan Kata Kunci MBG: ")
print("—"*45)

output_search = cari_relevansi(kata_kunci)
print(f"\nTop-{len(output_search)} Dokumen Paling Relevan untuk: '{kata_kunci}'")

# Menampilkan hasil dengan gradasi biru pada skor
display(output_search[['Peringkat', 'ID', 'Isi Dokumen', 'Skor_Kemiripan']].style.hide(axis='index').background_gradient(cmap='Blues', subset=['Skor_Kemiripan']))


—————————————————————————————————————————————
Masukkan Kata Kunci MBG: anak
—————————————————————————————————————————————

Top-3 Dokumen Paling Relevan untuk: 'anak'


Peringkat,ID,Isi Dokumen,Skor_Kemiripan
1,4,Seluruh anak sekolah harus mendapatkan asupan protein harian,0.319300
2,3,Kualitas nutrisi sangat penting untuk pertumbuhan anak sekolah,0.319300
3,2,Program makan bergizi gratis meningkatkan kualitas nutrisi siswa,0.000000
